# The prompt registry — version & promote prompts

*Part of the `gen_ai/` track. Prerequisite: `c_genai_evaluation` (we reuse its LLM-as-judge to pick a winner). The traditional-ML analog is `ml/f_model_registry`.*

## The problem: "which prompt was the good one?"

In `a_`/`c_`/`d_` the prompt was a **string literal buried in code**. The moment you start iterating — add a constraint, reword an instruction, tighten the format — you face the GenAI version of *"which pickle was the good model?"*: which prompt text produced which result, and is the new one actually better?

The **MLflow Prompt Registry** answers that. It version-controls prompt templates, lets you attach **aliases** (e.g. `@production`), and decouples the prompt from the code that runs it — the exact parallel of the **Model Registry** (`ml/f_model_registry`), but for prompts.

| Term | One-line meaning |
|---|---|
| **prompt** | A named, registered template with `{{variables}}` filled in at call time. |
| **version** | Each `register_prompt` of the same name appends an immutable version (1, 2, …). |
| **alias** | A movable label (e.g. `@production`) pointing at a version — what your app loads, so you re-point instead of editing code. |

## Prerequisites

- A **tracking server on port 5001** — the prompt registry lives in its registry store.
- **Ollama** (`qwen3:8b`) to run the prompts, and a **capable judge** (Azure, as in `c_`) to compare versions. Credentials from the gitignored `.env`.

No new dependencies — the prompt registry is part of `mlflow.genai`.

In [1]:
import os, re
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(), override=True)

# Judge config (same mapping as c_genai_evaluation): MLflow's azure provider reads AZURE_API_*.
os.environ["AZURE_API_KEY"]     = os.environ["AZURE_OPENAI_API_KEY"]
os.environ["AZURE_API_BASE"]    = os.environ["AZURE_OPENAI_BASE_URL"].removesuffix("/openai").rstrip("/")
os.environ["AZURE_API_VERSION"] = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21")
JUDGE = f"azure:/{os.environ.get('AZURE_OPENAI_LIGHT_MODEL', 'gpt-5.4-nano')}"

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-prompt-registry")

# Local model to actually run the prompts.
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=60, max_retries=0)
MODEL = "qwen3:8b"
PROMPT = "qa-answer"  # the registered prompt's name

2026/06/05 16:33:27 INFO mlflow.tracking.fluent: Experiment with name 'genai-prompt-registry' does not exist. Creating a new experiment.


## Register version 1

`register_prompt` stores a template under a name. Variables use **`{{double-brace}}`** syntax and are filled with `.format(...)` at call time. A first registration creates **version 1**.

In [2]:
v1 = mlflow.genai.register_prompt(
    name=PROMPT,
    template="Answer the user's question.\n\nQuestion: {{question}}",
    commit_message="v1: bare instruction",
)
print(f"registered {v1.name} v{v1.version} | variables: {v1.variables}")

2026/06/05 16:33:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: qa-answer, version 1


registered qa-answer v1 | variables: {'question'}


## Iterate → version 2

You try to do better: ask for a single concise sentence and a graceful "I don't know." Registering the **same name** appends version 2 — version 1 stays untouched and reproducible.

In [3]:
v2 = mlflow.genai.register_prompt(
    name=PROMPT,
    template=(
        "Answer the user's question in a single, concise, factual sentence. "
        "If you are unsure, say you don't know.\n\nQuestion: {{question}}"
    ),
    commit_message="v2: require one concise sentence",
)
print(f"now at v{v2.version}")

2026/06/05 16:33:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: qa-answer, version 2


now at v2


## Point an alias at a version

An **alias** is the stable handle your app loads. Set `@production` to a version and load it by URI — when you later promote a new version, you move the alias, and every app picks it up with **no code change**.

In [4]:
mlflow.genai.set_prompt_alias(name=PROMPT, alias="production", version=1)  # start on v1

loaded = mlflow.genai.load_prompt(f"prompts:/{PROMPT}@production")
print("production ->", f"v{loaded.version}")
print(loaded.format(question="What is MLflow Tracing?"))

production -> v1
Answer the user's question.

Question: What is MLflow Tracing?


## Run the prompt — loaded, not hard-coded

The app loads `@production`, fills in the question, and calls the model. The prompt text now lives in the registry, not in this function — swapping prompts is an alias move, not an edit.

In [5]:
def make_app(prompt):
    """Build a predict_fn that runs `prompt` (a loaded PromptVersion)."""
    def app(question: str) -> str:
        rendered = prompt.format(question=question)
        r = client.chat.completions.create(
            model=MODEL, max_tokens=512,
            messages=[{"role": "user", "content": rendered + " /no_think"}],
        )
        text = r.choices[0].message.content
        return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return app

print(make_app(loaded)("What is an MLflow model registry?"))

The **MLflow Model Registry** is a centralized model management system within the MLflow ecosystem that streamlines the process of tracking, versioning, and deploying machine learning models. It acts as a single source of truth for models across the machine learning lifecycle, enabling teams to manage models efficiently while ensuring consistency, collaboration, and reproducibility.

### Key Features of the MLflow Model Registry:
1. **Model Versioning**:
   - Tracks


## Compare versions and promote the winner

This is the payoff, and it ties back to `c_genai_evaluation`: run the **same questions** through each prompt version, score both with the LLM-as-judge, and promote whichever wins — the prompt-level version of `f_model_registry`'s champion/challenger gate.

In [6]:
from mlflow.genai.scorers import RelevanceToQuery, Guidelines

questions = [
    {"inputs": {"question": "What is MLflow Tracing?"}},
    {"inputs": {"question": "What is an MLflow model registry?"}},
    {"inputs": {"question": "What does mlflow models serve do?"}},
]
scorers = [
    RelevanceToQuery(model=JUDGE),
    Guidelines(name="concise", guidelines="The response is a single concise sentence.", model=JUDGE),
]

p1 = mlflow.genai.load_prompt(PROMPT, version=1)
p2 = mlflow.genai.load_prompt(PROMPT, version=2)
m1 = mlflow.genai.evaluate(data=questions, predict_fn=make_app(p1), scorers=scorers).metrics
m2 = mlflow.genai.evaluate(data=questions, predict_fn=make_app(p2), scorers=scorers).metrics
print("v1:", {k: round(v, 2) for k, v in m1.items()})
print("v2:", {k: round(v, 2) for k, v in m2.items()})

# Promote on the 'concise' guideline (v2 was written to win it).
winner = 2 if m2["concise/mean"] >= m1["concise/mean"] else 1
mlflow.genai.set_prompt_alias(name=PROMPT, alias="production", version=winner)
print(f"promoted v{winner} to @production")

MlflowException: The `version` argument should not be specified when loading a prompt by URI.

## Inspect what's registered

The UI shows it under **Prompts**, with each version's template, commit message, and aliases. Programmatically:

In [ ]:
for p in mlflow.genai.search_prompts():
    print(p.name)

prod = mlflow.genai.load_prompt(f"prompts:/{PROMPT}@production")
print(f"@production is now v{prod.version}; aliases on it: {prod.aliases}")

## Next steps

- **`f_genai_app_serving.ipynb`** (planned) — log the `d_` agent (or a prompt-driven app) as an MLflow **model** and serve it over REST; the served app can `load_prompt("prompts:/qa-answer@production")` so prompt changes ship by moving an alias, no redeploy.
- A registered prompt can also carry a **`response_format`** (a JSON schema) and **`model_config`** — register those alongside the template when your app needs structured output.